درج ذیل نوٹ بک GitHub Copilot Chat کی جانب سے خودکار طور پر تیار کی گئی ہے اور صرف ابتدائی سیٹ اپ کے لیے ہے


# پرامپٹ انجینئرنگ کا تعارف
پرامپٹ انجینئرنگ قدرتی زبان کی پروسیسنگ کے کاموں کے لیے پرامپٹس کو ڈیزائن اور بہتر بنانے کا عمل ہے۔ اس میں درست پرامپٹس کا انتخاب کرنا، ان کے پیرامیٹرز کو ایڈجسٹ کرنا، اور ان کی کارکردگی کا جائزہ لینا شامل ہے۔ NLP ماڈلز میں اعلیٰ درستی اور کارکردگی حاصل کرنے کے لیے پرامپٹ انجینئرنگ ناگزیر ہے۔ اس حصے میں، ہم OpenAI ماڈلز کا استعمال کرتے ہوئے پرامپٹ انجینئرنگ کی بنیادی باتوں کا جائزہ لیں گے۔


### مشق 1: ٹوکنائزیشن
tiktoken استعمال کرتے ہوئے ٹوکنائزیشن کی تلاش کریں، جو OpenAI کا ایک اوپن سورس تیز ٹوکنائزر ہے
مزید مثالوں کے لیے [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) دیکھیں۔


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### مشق 2: OpenAI API کلید کی سیٹنگ کی تصدیق کریں

نیچے دیا گیا کوڈ چلائیں تاکہ یہ تصدیق ہو سکے کہ آپ کا OpenAI اینڈ پوائنٹ صحیح طریقے سے سیٹ اپ ہے۔ کوڈ بس ایک سادہ بنیادی پرامپٹ آزماتا ہے اور تکمیل کی توثیق کرتا ہے۔ ان پٹ `oh say can you see` کی تکمیل کچھ اس طرح ہونی چاہیے `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### مشق 3: تخلیقات
دیکھیں کہ کیا ہوتا ہے جب آپ LLM سے ایسی پرامپٹ کے مکمل جوابات مانگتے ہیں جو ممکنہ طور پر موجود نہ ہو، یا ایسے موضوعات کے بارے میں جو اسے معلوم نہ ہوں کیونکہ وہ اس کے پری ٹرینڈ ڈیٹا سیٹ (زیادہ حالیہ) سے باہر ہیں۔ دیکھیں کہ جواب میں کیسے تبدیلی آتی ہے اگر آپ مختلف پرامپٹ یا مختلف ماڈل آزمائیں۔


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### مشق 4: ہدایت پر مبنی  
"text" متغیر کو بنیادی مواد مقرر کرنے کے لیے استعمال کریں  
اور "prompt" متغیر کو اس بنیادی مواد سے متعلق ہدایت فراہم کرنے کے لیے استعمال کریں۔  

یہاں ہم ماڈل سے درخواست کرتے ہیں کہ وہ متن کو دوسرے جماعت کے طالب علم کے لیے خلاصہ کرے  


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### مشق 5: پیچیدہ پرامپٹ
ایسا سوال آزمائیں جس میں نظام، صارف اور معاون کے پیغامات ہوں
نظام معاون کے سیاق و سباق کا تعین کرتا ہے
صارف اور معاون کے پیغامات کثیر مرحلہ گفتگو کا سیاق و سباق فراہم کرتے ہیں

نوٹ کریں کہ معاون کی شخصیت نظام کے سیاق و سباق میں "طنزیہ" مقرر کی گئی ہے۔
مختلف شخصیت کا سیاق و سباق آزما کر دیکھیں۔ یا مختلف قسم کے ان پٹ/آؤٹ پٹ پیغامات کا سلسلہ آزمائیں


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### مشق: اپنی اندرونی بصیرت کو دریافت کریں
اوپر دیے گئے مثالیں آپ کو ایسے نمونے دیتی ہیں جنہیں آپ نئے پرامپٹس (سادہ، پیچیدہ، ہدایتی وغیرہ) بنانے کے لیے استعمال کر سکتے ہیں - کچھ دیگر مشقیں بنانے کی کوشش کریں تاکہ ہم نے جن دیگر خیالات جیسے مثالیں، اشارے اور مزید بات کی ہے، ان کو دریافت کیا جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
